# Exploring the Dataset: Building the System CPU Metrics Table

**Goal:** Understand the structure of the `system.cpu` metricbeat logs, identify key fields, and map them to the `system_cpu_events` database table.

**Source file:** `gather/intranet_server/logs/2022-01-21-system_cpu.log`  
**What it contains:** CPU utilisation snapshots from the intranet server, collected every 45 seconds via metricbeat throughout 2022-01-21.

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below — everything else derives from it.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path(r"C:\Users\ishaanshetty\DATA-201\russellmitchell")

cpu_log_path = DATASET_ROOT / "gather" / "intranet_server" / "logs" / "2022-01-21-system_cpu.log"

print(f"Dataset found at: {DATASET_ROOT}")

Dataset found at: C:\Users\ishaanshetty\DATA-201\russellmitchell


## 1. Load the Raw CPU Log File

The file is a JSON Lines document — each line is one metricbeat CPU snapshot with deeply nested fields. We flatten it with `json_normalize` and rename the key fields to clean column names.

In [2]:
import json

import pandas as pd

raw_records = []
with open(cpu_log_path) as f:
    for line in f:
        raw_records.append(json.loads(line.strip()))

df_raw = pd.json_normalize(raw_records)

# Extract and rename the columns we care about
df = df_raw[
    [
        "@timestamp",
        "host.name",
        "system.cpu.total.pct",
        "system.cpu.user.pct",
        "system.cpu.system.pct",
        "system.cpu.idle.pct",
        "system.cpu.iowait.pct",
        "system.cpu.steal.pct",
        "system.cpu.softirq.pct",
        "system.cpu.cores",
        "event.duration",
        "metricset.period",
    ]
].copy()

df.columns = [
    "timestamp",
    "host",
    "cpu_total_pct",
    "cpu_user_pct",
    "cpu_system_pct",
    "cpu_idle_pct",
    "cpu_iowait_pct",
    "cpu_steal_pct",
    "cpu_softirq_pct",
    "cores",
    "event_duration_ns",
    "metricset_period_ms",
]

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df)} records")
print(f"\nColumns: {list(df.columns)}")
print(f"\nHost: {df['host'].iloc[0]}")
print(f"Date: {df['timestamp'].dt.date.iloc[0]}")
print(
    f"Time range: {df['timestamp'].min().strftime('%H:%M:%S')} UTC to {df['timestamp'].max().strftime('%H:%M:%S')} UTC"
)
print(
    f"Sampling interval: {df['metricset_period_ms'].iloc[0] / 1000:.0f}s  |  CPU cores: {int(df['cores'].iloc[0])}"
)

Loaded 1920 records

Columns: ['timestamp', 'host', 'cpu_total_pct', 'cpu_user_pct', 'cpu_system_pct', 'cpu_idle_pct', 'cpu_iowait_pct', 'cpu_steal_pct', 'cpu_softirq_pct', 'cores', 'event_duration_ns', 'metricset_period_ms']

Host: intranet-server
Date: 2022-01-21
Time range: 00:00:22 UTC to 23:59:37 UTC
Sampling interval: 45s  |  CPU cores: 1


## 2. Examine a Single Record

Before flattening, each record is a deeply nested JSON object from metricbeat. Here is what the raw structure looks like.

In [3]:
import json

print("Raw data for record 0:\n")
print(json.dumps(raw_records[0], indent=2))

Raw data for record 0:

{
  "agent": {
    "hostname": "intranet-server",
    "name": "intranet-server",
    "id": "d8e6f857-ec88-4cf5-bc5e-7b72a6fd1e33",
    "ephemeral_id": "c6b20ae2-38c0-4ada-8be2-2d607fddf7e9",
    "version": "7.13.2",
    "type": "metricbeat"
  },
  "service": {
    "type": "system"
  },
  "event": {
    "module": "system",
    "dataset": "system.cpu",
    "duration": 630326
  },
  "@version": "1",
  "metricset": {
    "period": 45000,
    "name": "cpu"
  },
  "host": {
    "cpu": {
      "pct": 0.0678
    },
    "name": "intranet-server"
  },
  "ecs": {
    "version": "1.9.0"
  },
  "@timestamp": "2022-01-21T00:00:22.284Z",
  "tags": [
    "beats_input_raw_event"
  ],
  "system": {
    "cpu": {
      "iowait": {
        "pct": 0.0002,
        "norm": {
          "pct": 0.0002
        }
      },
      "steal": {
        "pct": 0.0007,
        "norm": {
          "pct": 0.0007
        }
      },
      "irq": {
        "norm": {
          "pct": 0
        },
       

### What do these fields mean?

| Field | What It Is | Example |
|-------|------------|--------|
| `@timestamp` | When metricbeat collected this sample | `2022-01-21T00:00:22.284Z` |
| `system.cpu.total.pct` | Total CPU usage across all states | `0.0678` (6.8%) |
| `system.cpu.user.pct` | Time running user-space processes | `0.0283` |
| `system.cpu.system.pct` | Time in kernel / system calls | `0.0386` |
| `system.cpu.idle.pct` | Time the CPU was idle | `0.932` |
| `system.cpu.iowait.pct` | Time waiting for I/O to complete | `0.0002` |
| `system.cpu.steal.pct` | Time stolen by the hypervisor | `0.0007` |
| `event.duration` | How long metricbeat took to collect (ns) | `630326` |
| `metricset.period` | Configured collection interval (ms) | `45000` |

### 2.1 Record structure

Each line in the CPU log is a JSON object produced by metricbeat. The raw structure is deeply nested:
```json
{
  "@timestamp": "2022-01-21T00:00:22.284Z",
  "host": {"name": "intranet_server"},
  "system": {
    "cpu": {
      "total": {"pct": 0.0678},
      "user":  {"pct": 0.0283},
      ...
    }
  },
  "event": {"duration": 630326},
  "metricset": {"period": 45000}
}
```
`pd.json_normalize` flattens this into dotted column names (e.g. `system.cpu.total.pct`), which are then renamed to clean column names.


### 2.2 Parsing strategy

| Raw dotted field | Renamed column | Notes |
|-----------------|----------------|-------|
| `@timestamp` | `timestamp` | Parsed to `datetime64[ns, UTC]` |
| `host.name` | `host` | Server identifier string |
| `system.cpu.total.pct` | `cpu_total_pct` | Sum of all CPU states |
| `system.cpu.user.pct` | `cpu_user_pct` | User-space processes |
| `system.cpu.system.pct` | `cpu_system_pct` | Kernel/system calls |
| `system.cpu.idle.pct` | `cpu_idle_pct` | Idle time |
| `system.cpu.iowait.pct` | `cpu_iowait_pct` | Waiting for I/O |
| `system.cpu.steal.pct` | `cpu_steal_pct` | Hypervisor steal |
| `system.cpu.softirq.pct` | `cpu_softirq_pct` | Software interrupts |
| `system.cpu.cores` | `cores` | Logical core count |
| `event.duration` | `event_duration_ns` | Collection time in nanoseconds |
| `metricset.period` | `metricset_period_ms` | Configured interval in milliseconds |


## 3. Field-by-Field Exploration

### 3.1 timestamp


In [ ]:
print("=== Timestamp range ===")
print(f"  Earliest: {df['timestamp'].min()}")
print(f"  Latest:   {df['timestamp'].max()}")
print(f"  Span:     {df['timestamp'].max() - df['timestamp'].min()}")
print(f"  Records:  {len(df)}")
print(
    f"  Expected at 45s interval: {int((df['timestamp'].max() - df['timestamp'].min()).total_seconds() / 45) + 1}"
)
print()
dates = df["timestamp"].dt.date
for date, count in dates.value_counts().sort_index().items():
    print(f"  {date}: {count} records")

### 3.2 host


In [ ]:
print("=== host distribution ===")
for host, count in df["host"].value_counts().items():
    print(f"  {host}: {count}")

### 3.3 cpu pct fields


In [ ]:
pct_cols = [
    "cpu_total_pct",
    "cpu_user_pct",
    "cpu_system_pct",
    "cpu_idle_pct",
    "cpu_iowait_pct",
    "cpu_steal_pct",
    "cpu_softirq_pct",
]
print("=== CPU pct field ranges ===")
for col in pct_cols:
    print(
        f"  {col:20s}  min={df[col].min():.4f}  max={df[col].max():.4f}  mean={df[col].mean():.4f}  null={df[col].isna().sum()}"
    )

### 3.4 cores, event_duration_ns, metricset_period_ms


In [ ]:
print("=== cores ===")
print(df["cores"].value_counts().to_string())
print()
print("=== event_duration_ns ===")
print(df["event_duration_ns"].describe())
print()
print("=== metricset_period_ms ===")
print(df["metricset_period_ms"].value_counts().to_string())

## 4. Raw 1:1 DataFrame

The full unnormalized `df_raw` produced by `json_normalize` — one row per metricbeat snapshot, all original dotted field names preserved before renaming. This is the direct source for the `system_cpu_events` table.


In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print()
print("=== First 5 rows ===")
print(df_raw.head().to_string())
print()
print("=== Last 5 rows ===")
print(df_raw.tail().to_string())

## 5. Field Schema

After flattening and renaming, the working DataFrame has these columns. All `pct` values are expressed as a fraction of 1.0 (e.g. 0.06 = 6%).

In [4]:
schema_rows = [
    ("timestamp", "datetime64[ns, UTC]", "Metricbeat collection timestamp"),
    ("host", "str", "Hostname of the monitored server"),
    ("cpu_total_pct", "float64", "Total CPU usage (all states combined)"),
    ("cpu_user_pct", "float64", "CPU time in user-space processes"),
    ("cpu_system_pct", "float64", "CPU time in kernel/system calls"),
    ("cpu_idle_pct", "float64", "CPU idle time"),
    ("cpu_iowait_pct", "float64", "CPU waiting on I/O"),
    ("cpu_steal_pct", "float64", "CPU stolen by hypervisor"),
    ("cpu_softirq_pct", "float64", "CPU handling software interrupts"),
    ("cores", "int64", "Number of logical CPU cores"),
    ("event_duration_ns", "int64", "Time metricbeat took to collect (ns)"),
    ("metricset_period_ms", "int64", "Configured collection interval (ms)"),
]
pd.DataFrame(schema_rows, columns=["field", "type", "description"])

,field,type,description
0,timestamp,"datetime64[ns, UTC]",Metricbeat collection timestamp
1,host,str,Hostname of the monitored server
2,cpu_total_pct,float64,Total CPU usage (all states combined)
3,cpu_user_pct,float64,CPU time in user-space processes
4,cpu_system_pct,float64,CPU time in kernel/system calls
5,cpu_idle_pct,float64,CPU idle time
6,cpu_iowait_pct,float64,CPU waiting on I/O
7,cpu_steal_pct,float64,CPU stolen by hypervisor
8,cpu_softirq_pct,float64,CPU handling software interrupts
9,cores,int64,Number of logical CPU cores


## 6. Summary Statistics

Descriptive stats across the full day. Notice that the mean and median (50%) for `cpu_total_pct` are very close (~6–7%), indicating a stable baseline. The max of ~99% reveals the attack spike.

In [5]:
df[
    ["cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_idle_pct", "cpu_iowait_pct"]
].describe().round(4)

,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_idle_pct,cpu_iowait_pct
stat,,,,,
count,1920.0000,1920.0000,1920.0000,1920.0000,1920.0000
mean,0.0690,0.0302,0.0359,0.9295,0.0014
std,0.0622,0.0318,0.0124,0.0674,0.0078
min,0.0330,0.0179,0.0119,0.0000,0.0000
25%,0.0622,0.0270,0.0335,0.9322,0.0002
50%,0.0644,0.0282,0.0351,0.9348,0.0005
75%,0.0667,0.0294,0.0367,0.9372,0.0009
max,0.9984,0.7430,0.3104,0.9668,0.1443


## 7. Percentile Distribution

The p99 is still only 8.7%, showing that 99% of the day was normal. The spike sits entirely above the p99 threshold — it is a clear outlier.

In [6]:
pcts = [50, 75, 90, 95, 99, 100]
cols = ["cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_iowait_pct"]
rows = []
for p in pcts:
    row = {"percentile": f"p{p}"}
    for col in cols:
        row[col] = round(df[col].quantile(p / 100), 4)
    rows.append(row)
pd.DataFrame(rows)

,percentile,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,p50,0.0644,0.0282,0.0351,0.0005
1,p75,0.0667,0.0294,0.0367,0.0009
2,p90,0.0697,0.0308,0.0386,0.0024
3,p95,0.0728,0.0324,0.0406,0.0033
4,p99,0.0874,0.0366,0.0518,0.0099
5,p100,0.9984,0.7430,0.3104,0.1443


## 8. CPU Spike Records

Records where `cpu_total_pct > 0.5` (50%). All 10 fall within a 7-minute window starting at **06:26 UTC**. The high `cpu_user_pct` indicates the spike is driven by user-space processes — consistent with the webshell command execution seen in the access log at the same time.

In [7]:
df_spike = df[df["cpu_total_pct"] > 0.5][
    ["timestamp", "cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_iowait_pct"]
].reset_index(drop=True)
df_spike

,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,2022-01-21 06:26:37.284000+00:00,0.9938,0.0436,0.1090,0.0062
1,2022-01-21 06:27:22.284000+00:00,0.9984,0.0222,0.0940,0.0016
2,2022-01-21 06:28:07.284000+00:00,0.9976,0.1272,0.1097,0.0024
3,2022-01-21 06:28:52.284000+00:00,0.8904,0.7430,0.1217,0.1096
4,2022-01-21 06:29:37.284000+00:00,0.8490,0.4901,0.2233,0.1443
5,2022-01-21 06:30:22.284000+00:00,0.9513,0.5289,0.1616,0.0404
6,2022-01-21 06:31:07.284000+00:00,0.9559,0.5489,0.1783,0.0334
7,2022-01-21 06:31:52.284000+00:00,0.8916,0.5182,0.2683,0.0999
8,2022-01-21 06:32:37.284000+00:00,0.8492,0.5054,0.2370,0.1432
9,2022-01-21 06:33:22.284000+00:00,0.8502,0.4841,0.3104,0.1437


## 9. Spike in Context

The same window with 3 baseline samples before and after — shows the sharp rise from ~6% to ~99% and the clean return to normal once the attacker's commands finished.

In [8]:
spike_idx = df[df["cpu_total_pct"] > 0.5].index
ctx_start = max(0, spike_idx[0] - 3)
ctx_end = min(len(df) - 1, spike_idx[-1] + 3)

df.iloc[ctx_start : ctx_end + 1][
    ["timestamp", "cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_iowait_pct"]
].reset_index(drop=True)

,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,2022-01-21 06:24:22.284000+00:00,0.0624,0.0252,0.0362,0.0002
1,2022-01-21 06:25:07.284000+00:00,0.0749,0.0334,0.0407,0.1073
2,2022-01-21 06:25:52.284000+00:00,0.1162,0.0466,0.0679,0.1312
3,2022-01-21 06:26:37.284000+00:00,0.9938,0.0436,0.1090,0.0062
4,2022-01-21 06:27:22.284000+00:00,0.9984,0.0222,0.0940,0.0016
5,2022-01-21 06:28:07.284000+00:00,0.9976,0.1272,0.1097,0.0024
6,2022-01-21 06:28:52.284000+00:00,0.8904,0.7430,0.1217,0.1096
7,2022-01-21 06:29:37.284000+00:00,0.8490,0.4901,0.2233,0.1443
8,2022-01-21 06:30:22.284000+00:00,0.9513,0.5289,0.1616,0.0404
9,2022-01-21 06:31:07.284000+00:00,0.9559,0.5489,0.1783,0.0334


## 10. Mapping to the Database Schema

### 10.0 Type inference assumptions

All SQL type assignments are inferred from observed values. See `type_inference_assumptions.md` for shared team-level rules.

Key decisions for this file:
- `event_timestamp`: TIMESTAMP WITH TIME ZONE — metricbeat records include UTC offset
- `hostname`: TEXT — server name string, no fixed max length required
- All `cpu_*_pct` columns: NUMERIC(6,4) — values range 0.0–1.0 with 4 decimal places of precision; avoids floating-point rounding errors
- `cpu_cores`: SMALLINT — core count is a small integer (observed: 1)
- `event_duration_ns` and `metricset_period_ms` are retained for completeness but carry no analytical value in a single-source table

### 10.1 Field mapping

Here is how the flattened fields map to the planned `system_cpu_events` PostgreSQL table, followed by a preview of the first 10 rows as they will look once ingested.

| source_field | postgres_column | postgres_type | notes |
|---|---|---|---|
| timestamp | event_timestamp | TIMESTAMPTZ | Collection time from metricbeat |
| host | hostname | TEXT | Server identifier |
| cpu_total_pct | cpu_total_pct | NUMERIC(6,4) | 0.0–1.0 fraction |
| cpu_user_pct | cpu_user_pct | NUMERIC(6,4) | User-space fraction |
| cpu_system_pct | cpu_system_pct | NUMERIC(6,4) | Kernel fraction |
| cpu_idle_pct | cpu_idle_pct | NUMERIC(6,4) | Idle fraction |
| cpu_iowait_pct | cpu_iowait_pct | NUMERIC(6,4) | I/O wait fraction |
| cpu_steal_pct | cpu_steal_pct | NUMERIC(6,4) | Hypervisor steal fraction |
| cpu_softirq_pct | cpu_softirq_pct | NUMERIC(6,4) | Soft IRQ fraction |
| cores | cpu_cores | SMALLINT | Logical core count |

In [9]:
# Preview of the system_cpu_events table as it will look in PostgreSQL
db_preview = (
    df[
        [
            "timestamp",
            "cpu_total_pct",
            "cpu_user_pct",
            "cpu_system_pct",
            "cpu_idle_pct",
            "cpu_iowait_pct",
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)
db_preview.insert(0, "system_cpu_id", range(1, 11))
db_preview

,system_cpu_id,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_idle_pct,cpu_iowait_pct
0,1,2022-01-21 00:00:22.284000+00:00,0.0678,0.0283,0.0386,0.9320,0.0002
1,2,2022-01-21 00:01:07.284000+00:00,0.0439,0.0211,0.0221,0.9559,0.0002
2,3,2022-01-21 00:01:52.284000+00:00,0.0500,0.0224,0.0267,0.9500,0.0000
3,4,2022-01-21 00:02:37.284000+00:00,0.0596,0.0262,0.0326,0.9404,0.0000
4,5,2022-01-21 00:03:22.284000+00:00,0.0610,0.0270,0.0331,0.9385,0.0005
5,6,2022-01-21 00:04:07.284000+00:00,0.0664,0.0279,0.0380,0.9331,0.0005
6,7,2022-01-21 00:04:52.284000+00:00,0.0626,0.0278,0.0339,0.9372,0.0002
7,8,2022-01-21 00:05:37.288000+00:00,0.0644,0.0277,0.0357,0.9354,0.0002
8,9,2022-01-21 00:06:22.284000+00:00,0.0640,0.0266,0.0355,0.9360,0.0000
9,10,2022-01-21 00:07:07.284000+00:00,0.0643,0.0273,0.0360,0.9348,0.0009


In [ ]:
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE system_cpu_events (
    system_cpu_event_id SERIAL PRIMARY KEY,
    event_timestamp     TIMESTAMP WITH TIME ZONE NOT NULL,
    hostname            TEXT,
    cpu_total_pct       NUMERIC(6,4),
    cpu_user_pct        NUMERIC(6,4),
    cpu_system_pct      NUMERIC(6,4),
    cpu_idle_pct        NUMERIC(6,4),
    cpu_iowait_pct      NUMERIC(6,4),
    cpu_steal_pct       NUMERIC(6,4),
    cpu_softirq_pct     NUMERIC(6,4),
    cpu_cores           SMALLINT,
    created_at          TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

mysql_ddl = """
-- MySQL
CREATE TABLE system_cpu_events (
    system_cpu_event_id INT AUTO_INCREMENT PRIMARY KEY,
    event_timestamp     DATETIME NOT NULL,
    hostname            VARCHAR(255),
    cpu_total_pct       DECIMAL(6,4),
    cpu_user_pct        DECIMAL(6,4),
    cpu_system_pct      DECIMAL(6,4),
    cpu_idle_pct        DECIMAL(6,4),
    cpu_iowait_pct      DECIMAL(6,4),
    cpu_steal_pct       DECIMAL(6,4),
    cpu_softirq_pct     DECIMAL(6,4),
    cpu_cores           SMALLINT,
    created_at          DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""

print(postgresql_ddl)
print(mysql_ddl)

### 10.2 Raw DDL


In [ ]:
print("=== Summary ===")
print(f"Total records:        {len(df)}")
print(f"Time range:           {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Span:                 {df['timestamp'].max() - df['timestamp'].min()}")
print(f"Distinct hosts:       {df['host'].nunique()}")
print(f"CPU cores (observed): {df['cores'].unique()}")
print(f"Columns in df:        {len(df.columns)}")
print()

print("Columns with nulls:")
for col in df.columns:
    n = df[col].isna().sum()
    if n > 0:
        print(f"  {col}: {n} nulls ({n / len(df) * 100:.1f}%)")
print()

spike_count = (df["cpu_total_pct"] > 0.5).sum()
print(f"Spike records (>50% CPU): {spike_count} ({spike_count / len(df) * 100:.1f}%)")
print(
    f"Normal records (<50% CPU): {len(df) - spike_count} ({(len(df) - spike_count) / len(df) * 100:.1f}%)"
)

## 11. Normalization Observations

Applying the `normalization_rules_sheet.md` checklist to the raw system CPU metrics data.

### 11.1 1NF Check

**Atomic fields:** All columns after flattening with `json_normalize` are scalar values (timestamps, floats, integers, strings). No arrays or nested structures remain in the working DataFrame.

**Repeating groups:** None. The CPU percentage fields (`cpu_total_pct`, `cpu_user_pct`, `cpu_system_pct`, etc.) are separate named columns, not a repeating group — each measures a distinct CPU state.

**1NF status: satisfied.** All fields are atomic after flattening.

### 11.2 2NF Check

**Primary key:** `system_cpu_event_id` is a single-column surrogate PK. Partial dependencies require a composite key, which does not exist here.

**2NF status: satisfied.** Single-column PK makes partial dependencies impossible.

### 11.3 3NF Check

**Transitive dependencies identified:**

| Determinant | Dependent(s) | Notes |
|-------------|-------------|-------|
| `hostname` | server configuration | The hostname could determine `cpu_cores` if cores are fixed per server. In this dataset there is only one host, so this cannot be verified. |
| `cpu_total_pct` | derived sum | `cpu_total_pct` is approximately the sum of `cpu_user_pct + cpu_system_pct + cpu_iowait_pct + cpu_steal_pct + cpu_softirq_pct`. It is a computed aggregate, not independently measured. |

**3NF status: acceptable.** `cpu_total_pct` is technically derivable from the other pct columns but is stored for query convenience. The `hostname -> cpu_cores` dependency is deferred — if multiple hosts are added, `cpu_cores` should move to a hosts dimension table.

### 11.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | system_cpu_event_id | all attributes | Surrogate PK, trivially determines everything. |
| FD2 | event_timestamp | all metrics | Timestamps are unique per collection interval (45s). Candidate natural key. |
| FD3 | hostname | cpu_cores | If cores are fixed per host, hostname functionally determines cpu_cores. Relevant when multiple hosts are added. |
| FD4 | cpu_user_pct, cpu_system_pct, cpu_iowait_pct, cpu_steal_pct, cpu_softirq_pct | cpu_total_pct | Total is the sum of components. Stored denormalized for convenience. |


## 12. Key Findings for Schema Design

1. **Clean, flat structure:** After `json_normalize`, all metricbeat fields are scalar. No 1NF violations — this is the simplest schema of all the log sources explored.

2. **`cpu_total_pct` is a derived column:** It is approximately the sum of the other pct fields. Stored for query convenience but could be dropped in a strict normalized schema and computed as a view.

3. **Attack spike is a clear outlier:** 99% of the day has `cpu_total_pct` below 8.7% (p99). The 10 spike records (>50% CPU) all fall within a 7-minute window at 06:26 UTC, directly correlating with webshell command execution seen in the access log.

4. **Single host, single day:** This file covers only `intranet_server` on 2022-01-21. When additional hosts or days are added, `hostname` should become a FK to a hosts dimension table, and `cpu_cores` should move there too.

5. **Collection interval is fixed:** `metricset_period_ms` is always 45,000 ms. This column adds no analytical value in a single-source table and could be dropped or moved to a metadata table.

6. **No label integration:** Unlike the Apache logs, CPU records have no direct annotation labels. Attack detection relies on threshold comparison (`cpu_total_pct > 0.5`) cross-referenced with timestamps from the access log.

7. **NUMERIC(6,4) is appropriate:** All pct values range 0.0–1.0 with 4 decimal places of precision in the source data. `NUMERIC(6,4)` covers the full range without floating-point rounding errors.
